# 69 — Expense Processing Agent

## Goal
Build an agent that **reads receipts** (text-based), **categorizes
expenses**, **flags unusual claims**, and **drafts a reimbursement
summary**.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Receipt parsing** | Extract structured data from receipt text |
| **Expense categorization** | Map items to budget categories |
| **Anomaly flagging** | Flag expenses above thresholds |
| **Multi-step workflow** | Parse → Categorize → Validate → Summarize |
| **OCR concept** | Text-based proxy for real OCR pipeline |

## Architecture
```
Receipt text → Parse → Categorize → Validate (flag anomalies) → Reimbursement report
```

## Stack
- `LangGraph` + `ChatOllama`
- Custom expense tools
- Simulated receipt OCR output

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from datetime import datetime
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

print("All imports OK")

## Step 1 — Simulated Receipts (OCR Output)

In production, an OCR engine (PaddleOCR, Tesseract) would
extract text from receipt images. We simulate that output.

In [ ]:
RECEIPTS = {
    "R001": {
        "image": "receipt_001.jpg",
        "ocr_text": """HILTON GARDEN INN
San Francisco, CA
Date: 2025-06-10
Check-in: Jun 10  Check-out: Jun 12
Room: Standard King - 2 nights
Rate: $189.00/night
Subtotal: $378.00
Tax (14.5%): $54.81
Total: $432.81
Payment: Visa ****4532
Guest: Sarah Mitchell""",
        "submitter": "Sarah Mitchell",
        "trip": "Client meeting SF",
    },
    "R002": {
        "image": "receipt_002.jpg",
        "ocr_text": """UBER RECEIPT
Trip: SFO Airport → Hilton Garden Inn
Date: June 10, 2025
UberX
Distance: 13.2 mi
Duration: 28 min
Fare: $34.50
Tip: $7.00
Total: $41.50
Rider: Sarah Mitchell""",
        "submitter": "Sarah Mitchell",
        "trip": "Client meeting SF",
    },
    "R003": {
        "image": "receipt_003.jpg",
        "ocr_text": """THE HOUSE RESTAURANT
1230 Grant Ave, San Francisco
Date: 06/10/2025
Table: 4  Server: Mike

2x Wagyu Steak        $89.00 each  $178.00
1x Lobster Risotto                  $45.00
3x Craft Cocktail     $18.00 each   $54.00
1x Wine Bottle (Opus One 2019)     $450.00
Subtotal:                          $727.00
Tax (8.625%):                       $62.70
Tip (22%):                         $159.94
TOTAL:                             $949.64
Card: Amex ****1234
Guests: 3""",
        "submitter": "Sarah Mitchell",
        "trip": "Client meeting SF",
    },
    "R004": {
        "image": "receipt_004.jpg",
        "ocr_text": """OFFICE DEPOT
Online Order #78432
Date: 2025-06-08

HP LaserJet Toner (2-pack)    $129.99
Copy Paper 10-Ream Case        $49.99
USB-C Hub                      $34.99
Shipping:                       $0.00
Tax:                           $17.75
TOTAL:                        $232.73
Delivered to: Main Office""",
        "submitter": "James Parker",
        "trip": None,
    },
    "R005": {
        "image": "receipt_005.jpg",
        "ocr_text": """BEST BUY
Store #1142
Date: 2025-06-05

Apple MacBook Pro 16\"   $3,499.00
AppleCare+ 3yr            $399.00
Tax:                      $321.91
TOTAL:                  $4,219.91
Card: Visa ****8877
Note: Personal laptop upgrade""",
        "submitter": "Dave Wilson",
        "trip": None,
    },
}

# Expense policy limits
POLICY = {
    "hotel": {"daily_limit": 250, "requires_approval_above": 200},
    "meals": {"per_meal_limit": 75, "daily_limit": 150, "alcohol_policy": "not reimbursable"},
    "transport": {"ride_limit": 75, "daily_limit": 150},
    "office_supplies": {"single_item_limit": 500, "requires_approval_above": 200},
    "equipment": {"requires_pre_approval": True, "limit": 2000},
}

_expense_reports = []

print(f"Loaded {len(RECEIPTS)} receipts")
print(f"Policy categories: {list(POLICY.keys())}")

## Step 2 — Define Expense Processing Tools

In [ ]:
@tool
def get_receipt(receipt_id: str) -> str:
    """Get the OCR-extracted text from a receipt image."""
    if receipt_id not in RECEIPTS:
        return f"Receipt {receipt_id} not found. Available: {list(RECEIPTS.keys())}"
    r = RECEIPTS[receipt_id]
    return f"Receipt: {receipt_id}\nSubmitter: {r['submitter']}\nTrip: {r['trip'] or 'N/A'}\n\nOCR Text:\n{r['ocr_text']}"

@tool
def get_expense_policy() -> str:
    """Get the company expense reimbursement policy with limits and rules."""
    return json.dumps(POLICY, indent=2)

@tool
def list_pending_receipts() -> str:
    """List all pending receipts awaiting processing."""
    lines = ["Pending receipts:"]
    for rid, r in RECEIPTS.items():
        lines.append(f"  {rid}: {r['submitter']} — {r['trip'] or 'office/other'}")
    return "\n".join(lines)

@tool
def save_expense_report(report_json: str) -> str:
    """Save a processed expense report. Must include: receipt_id, submitter, category, amount, status (approved/flagged/denied), flags (list of issues), notes."""
    try:
        report = json.loads(report_json)
        required = ["receipt_id", "submitter", "category", "amount", "status", "flags", "notes"]
        missing = [k for k in required if k not in report]
        if missing:
            return f"Missing fields: {missing}"
        _expense_reports.append(report)
        status_icon = "✅" if report["status"] == "approved" else "⚠️" if report["status"] == "flagged" else "❌"
        return f"{status_icon} Expense report saved: {report['receipt_id']} — {report['status']}"
    except json.JSONDecodeError as e:
        return f"Invalid JSON: {e}"

@tool 
def generate_reimbursement_summary() -> str:
    """Generate a final reimbursement summary from all processed expense reports."""
    if not _expense_reports:
        return "No expense reports to summarize."
    
    approved_total = sum(r["amount"] for r in _expense_reports if r["status"] == "approved")
    flagged_total = sum(r["amount"] for r in _expense_reports if r["status"] == "flagged")
    denied_total = sum(r["amount"] for r in _expense_reports if r["status"] == "denied")
    
    summary = f"""REIMBURSEMENT SUMMARY
{'='*50}
Total Reports: {len(_expense_reports)}
Approved: ${approved_total:,.2f}
Flagged (pending review): ${flagged_total:,.2f}
Denied: ${denied_total:,.2f}
{'='*50}

Detail:"""
    for r in _expense_reports:
        icon = "✅" if r["status"] == "approved" else "⚠️" if r["status"] == "flagged" else "❌"
        summary += f"\n  {icon} {r['receipt_id']}: ${r['amount']:,.2f} [{r['category']}] — {r['status']}"
        if r["flags"]:
            summary += f"\n     Flags: {', '.join(r['flags'])}"
    return summary

expense_tools = [get_receipt, get_expense_policy, list_pending_receipts, save_expense_report, generate_reimbursement_summary]
print(f"Defined {len(expense_tools)} expense tools")

## Step 3 — Create the Expense Agent

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0)

EXPENSE_PROMPT = """You are an expense processing assistant. Your job is to:

1. Read receipt OCR text using get_receipt()
2. Check the expense policy using get_expense_policy()
3. Categorize the expense (hotel, meals, transport, office_supplies, equipment)
4. Extract the total amount
5. Check for policy violations and flag issues:
   - Over daily/per-item limits
   - Alcohol on meal receipts (not reimbursable)
   - Equipment without pre-approval
   - Unusually high amounts
   - Personal expenses
6. Save the expense report with appropriate status

Status rules:
- 'approved': Within policy, no issues
- 'flagged': Has policy concerns, needs manager review
- 'denied': Clear policy violation (personal expense, no business purpose)

Always explain your reasoning for the status decision."""

expense_agent = create_react_agent(
    model=llm,
    tools=expense_tools,
    prompt=EXPENSE_PROMPT,
)

def process_expense(request: str):
    print(f"\n{'='*70}")
    print(f"REQUEST: {request}")
    print(f"{'='*70}")
    result = expense_agent.invoke({"messages": [{"role": "user", "content": request}]})
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if role == "AIMessage" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🔧 TOOL: {tc['name']}({str(tc['args'])[:100]})")
        elif role == "ToolMessage":
            print(f"📋 RESULT: {msg.content[:400]}")
        elif role == "AIMessage" and msg.content:
            print(f"\n🤖 ASSISTANT:\n{msg.content}")
    return result

print("Expense agent ready!")

In [ ]:
# Process a clean hotel receipt
process_expense("Process receipt R001 (hotel stay).")

In [ ]:
# Process the expensive restaurant receipt (should flag alcohol + high amount)
process_expense("Process receipt R003 (client dinner). Check for policy violations.")

In [ ]:
# Process the personal MacBook receipt (should deny)
process_expense("Process receipt R005.")

In [ ]:
# Generate the final reimbursement summary
process_expense("Generate the final reimbursement summary for all processed receipts.")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Receipt OCR** | Simulated OCR text extraction from receipts |
| **Policy engine** | JSON rules checked against parsed amounts |
| **Anomaly flagging** | Agent identifies policy violations |
| **Status decisions** | Approved / Flagged / Denied with reasoning |
| **Summary report** | Aggregated reimbursement totals |

## Real OCR Integration
```python
# Replace simulated OCR with real PaddleOCR:
from paddleocr import PaddleOCR

ocr = PaddleOCR(use_angle_cls=True, lang='en')
result = ocr.ocr('receipt_001.jpg', cls=True)
text = '\n'.join([line[1][0] for line in result[0]])
```

## LangGraph Extension
```
For production, add:
- Manager approval node (interrupt + resume)
- Auto-categorization confidence score
- Integration with accounting system (QuickBooks, SAP)
- Batch processing of multiple receipts
```